In [7]:
import ee
import geemap
import datetime
import time

hello


In [ ]:
ee.Authenticate()
ee.Initialize(project='thesis-478211')

In [ ]:
START_YEAR = 2016
END_YEAR = datetime.datetime.now().year
SCALE = 10
MAX_PIXELS = 1e13
DRIVE_FOLDER = "DW_Quarterly_Colorized"   

boundary = ee.FeatureCollection("projects/thesis-478211/assets/BC_Boundary")

dw = ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1").filterBounds(boundary)

VIS_PALETTE = [
    '419bdf','397d49','88b053','7a87c6','e49635',
    'dfc35a','c4281b','a59b8f','b39fe1'
]

def quarterly_dw_rgb(year, quarter):
    start_month = quarter * 3 + 1
    start = ee.Date.fromYMD(year, start_month, 1)
    end = start.advance(3, "month")

    filtered = dw.filterDate(start, end).select("label")

    count = filtered.size().getInfo()
    if count == 0:
        return None

    img_mode = filtered.mode().clip(boundary)

    img_rgb = img_mode.visualize(min=0, max=8, palette=VIS_PALETTE)
    img_rgb = img_rgb.set("year", year).set("quarter", quarter + 1)

    return img_rgb

exports = []

for year in range(START_YEAR, END_YEAR + 1):
    for quarter in range(4):
        print(f"Processing {year} Q{quarter+1} ...")

        img_rgb = quarterly_dw_rgb(year, quarter)

        if img_rgb is None:
            print(f"Skipping {year} Q{quarter+1} — no DW data")
            continue

        description = f"DW_RGB_{year}_Q{quarter+1}"

        task = ee.batch.Export.image.toDrive(
            image = img_rgb,
            description = description,
            folder = DRIVE_FOLDER,
            fileNamePrefix = description,
            region = boundary.geometry(),
            scale = SCALE,
            maxPixels = MAX_PIXELS
        )

        task.start()
        exports.append((description, task))
        print(f"Started export: {description}")

        while True:
            status = task.status()
            state = status['state']

            if state in ['COMPLETED', 'FAILED', 'CANCELLED']:
                break
            print(f"  Task status: {state} ... waiting 30s")
            time.sleep(30) 

        if state == 'COMPLETED':
            print(f"Export successful: {description}")
        else:
            print(f"Export {state}: {description}")

print("\n=== ALL EXPORTS FINISHED ===")
for desc, task in exports:
    status = task.status()['state']
    print(f"{desc}: {status}")